<a href="https://colab.research.google.com/github/Frans1828/scikit-learn-cookbook/blob/main/12.%20Cross-Validation%20and%20Model%20Evaluation/ch12_crossval_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
## Identitas Mahasiswa

| Informasi | Keterangan |
|---|---|
| **Nama** | Fransisco Sitepu |
| **NIM** | 101032300004 |
| **Kelas** | TK 47 03 |
| **Mata Kuliah** | Machine Learning |
| **Institusi** | Telkom University |

---


# Chapter 12: Cross-Validation and Model Evaluation

## Mengapa Evaluasi yang Tepat Sangat Penting?

Evaluasi model yang salah dapat menghasilkan kesimpulan yang menyesatkan. Masalah umum:
- **Overfitting pada test set**: jika model dipilih berdasarkan performa test set, maka test set tidak lagi memberikan estimasi yang tidak bias
- **Data leakage**: preprocessing yang dilakukan sebelum split dapat menyebabkan informasi test data bocor ke training
- **High variance estimasi**: single train-test split memberikan estimasi yang bergantung pada pembagian data

## K-Fold Cross-Validation

K-Fold CV membagi data menjadi $K$ bagian (folds) yang sama besar:
1. Untuk setiap fold $k = 1, ..., K$:
   - **Test set**: fold ke-$k$
   - **Training set**: semua fold lainnya
2. Hitung skor pada setiap fold
3. Estimasi performa: $\bar{s} \pm \sigma$

**Bias-Variance tradeoff dalam K**:
- **K kecil** (misal K=2): estimasi bias tinggi, variance rendah
- **K besar** (misal K=n, LOOCV): estimasi bias rendah, variance tinggi, mahal secara komputasi
- **K=5 atau K=10**: kompromi terbaik dalam praktik

### Stratified K-Fold

Untuk klasifikasi, penting menjaga **proporsi kelas** yang sama di setiap fold. `StratifiedKFold` melakukan ini secara otomatis.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris, make_classification
from sklearn.model_selection import (
    KFold, StratifiedKFold, cross_val_score, cross_validate,
    GridSearchCV, RandomizedSearchCV,
    learning_curve, validation_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    make_scorer
)
from scipy.stats import uniform, randint

cancer = load_breast_cancer()
X, y = cancer.data, cancer.target
iris = load_iris()
Xi, yi = iris.data, iris.target

print(f'Cancer: {X.shape}, classes={cancer.target_names}')
print(f'Iris: {Xi.shape}, classes={iris.target_names}')


In [ ]:
# Demonstrating K-Fold vs Stratified K-Fold
from sklearn.model_selection import KFold, StratifiedKFold

print('=== Comparing K-Fold vs StratifiedKFold ===')
print('Class distribution in each fold:\n')

for CVClass, name in [(KFold(n_splits=5, shuffle=True, random_state=42), 'KFold'),
                       (StratifiedKFold(n_splits=5, shuffle=True, random_state=42), 'StratifiedKFold')]:
    print(f'{name}:')
    for fold, (tr_idx, te_idx) in enumerate(CVClass.split(X, y)):
        n_pos = y[te_idx].sum()
        n_neg = len(te_idx) - n_pos
        ratio = n_pos / len(te_idx)
        print(f'  Fold {fold+1}: test size={len(te_idx)}, positive={n_pos} ({ratio:.1%}), negative={n_neg}')
    print()

# Global class distribution
print(f'Overall positive rate: {y.mean():.1%}')


In [ ]:
# cross_validate: multiple metrics at once
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42))
])

scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score),
    'recall': make_scorer(recall_score),
    'f1': make_scorer(f1_score),
    'roc_auc': make_scorer(roc_auc_score)
}

cv_results = cross_validate(
    pipe_rf, X, y,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring=scoring,
    return_train_score=True
)

print('=== Random Forest 5-Fold CV Results ===')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    train_mean = cv_results[f'train_{metric}'].mean()
    test_mean = cv_results[f'test_{metric}'].mean()
    test_std = cv_results[f'test_{metric}'].std()
    print(f'{metric:<12}: train={train_mean:.4f}, test={test_mean:.4f} (+/-{test_std:.4f})')


## 12.2 Learning Curves

Learning curve menunjukkan bagaimana performa model berubah seiring bertambahnya training data:

- **Underfitting (high bias)**: training dan validation scores keduanya rendah dan mendekati satu sama lain
- **Overfitting (high variance)**: training score tinggi, validation score jauh lebih rendah
- **Good fit**: keduanya konvergen ke performa yang tinggi

Jika ada gap besar antara training dan validation, tambahkan lebih banyak data. Jika keduanya plateau, tambahkan fitur atau ganti model.


In [ ]:
# Learning curves for multiple models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_lc = [
    ('Underfitting\n(Logistic Reg, C=0.001)',
     Pipeline([('scaler', StandardScaler()),
               ('lr', LogisticRegression(C=0.001, max_iter=1000))])),
    ('Good Fit\n(Random Forest)',
     Pipeline([('scaler', StandardScaler()),
               ('rf', RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42))])),
    ('Overfitting\n(Decision Tree, deep)',
     Pipeline([('scaler', StandardScaler()),
               ('rf', RandomForestClassifier(n_estimators=10, max_depth=None, random_state=42))]))
]

cv_lc = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_sizes = np.linspace(0.1, 1.0, 10)

for ax, (title, model) in zip(axes, models_lc):
    sizes, train_scores, val_scores = learning_curve(
        model, X, y,
        train_sizes=train_sizes,
        cv=cv_lc, scoring='accuracy',
        n_jobs=-1
    )
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)

    ax.plot(sizes, train_mean, 'b-o', lw=2, markersize=5, label='Training score')
    ax.fill_between(sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='blue')
    ax.plot(sizes, val_mean, 'r-s', lw=2, markersize=5, label='Validation score')
    ax.fill_between(sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='red')
    ax.set_xlabel('Training set size')
    ax.set_ylabel('Accuracy')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()


## 12.3 Hyperparameter Tuning: GridSearchCV vs RandomizedSearchCV

### GridSearchCV

Evaluasi setiap kombinasi hyperparameter secara exhaustif. Untuk grid dengan $N_1 \times N_2 \times ... \times N_k$ kombinasi dan $K$-fold CV, total model yang dilatih:

$$\text{Total fits} = N_1 \times N_2 \times ... \times N_k \times K$$

### RandomizedSearchCV

Sampling acak dari distribusi hyperparameter. Lebih efisien untuk ruang pencarian besar:

$$\text{Total fits} = n\_iter \times K$$

Berguna karena tidak semua hyperparameter sama pentingnya — random search lebih sering menemukan kombinasi yang baik dengan lebih sedikit evaluasi.


In [ ]:
import time

cv_hp = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# GridSearchCV
grid_param = {
    'rf__n_estimators': [50, 100, 200],
    'rf__max_depth': [3, 5, 7, None],
    'rf__min_samples_leaf': [1, 2, 4]
}
pipe_hp = Pipeline([
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

start = time.time()
grid_search = GridSearchCV(pipe_hp, grid_param, cv=cv_hp, scoring='roc_auc', n_jobs=-1)
grid_search.fit(X, y)
grid_time = time.time() - start

print(f'GridSearchCV:')
print(f'  Combinations: {len(grid_search.cv_results_["params"])}')
print(f'  Time: {grid_time:.2f}s')
print(f'  Best params: {grid_search.best_params_}')
print(f'  Best AUC: {grid_search.best_score_:.4f}')

# RandomizedSearchCV
random_param = {
    'rf__n_estimators': randint(50, 300),
    'rf__max_depth': [3, 5, 7, 10, None],
    'rf__min_samples_leaf': randint(1, 10),
    'rf__max_features': uniform(0.3, 0.7)
}

start = time.time()
rand_search = RandomizedSearchCV(
    pipe_hp, random_param, n_iter=30,
    cv=cv_hp, scoring='roc_auc', n_jobs=-1, random_state=42
)
rand_search.fit(X, y)
rand_time = time.time() - start

print(f'\nRandomizedSearchCV:')
print(f'  Evaluations: 30')
print(f'  Time: {rand_time:.2f}s')
print(f'  Best params: {rand_search.best_params_}')
print(f'  Best AUC: {rand_search.best_score_:.4f}')


In [ ]:
# Validation Curve: effect of a single hyperparameter
pipe_vc = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

C_range = np.logspace(-3, 3, 10)
train_scores_vc, val_scores_vc = validation_curve(
    pipe_vc, X, y,
    param_name='svm__C',
    param_range=C_range,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
    n_jobs=-1
)

fig, ax = plt.subplots(figsize=(9, 5))
ax.semilogx(C_range, train_scores_vc.mean(axis=1), 'b-o', lw=2, label='Training score')
ax.fill_between(C_range, train_scores_vc.mean(axis=1) - train_scores_vc.std(axis=1),
                train_scores_vc.mean(axis=1) + train_scores_vc.std(axis=1), alpha=0.15, color='blue')
ax.semilogx(C_range, val_scores_vc.mean(axis=1), 'r-s', lw=2, label='Validation score')
ax.fill_between(C_range, val_scores_vc.mean(axis=1) - val_scores_vc.std(axis=1),
                val_scores_vc.mean(axis=1) + val_scores_vc.std(axis=1), alpha=0.15, color='red')
best_C_idx = val_scores_vc.mean(axis=1).argmax()
ax.axvline(x=C_range[best_C_idx], color='green', linestyle='--', label=f'Best C={C_range[best_C_idx]:.3f}')
ax.set_xlabel('C (Regularization Strength)')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Curve: SVM C parameter (Breast Cancer)')
ax.legend()
plt.tight_layout()
plt.show()
print(f'Best C: {C_range[best_C_idx]:.3f}, Validation Acc: {val_scores_vc.mean(axis=1)[best_C_idx]:.4f}')


## Ringkasan: Key Takeaways Chapter 12

| Konsep | Poin Kunci |
|---|---|
| **K-Fold CV** | Estimasi performa yang lebih reliable; gunakan K=5 atau K=10 dalam praktik |
| **Stratified K-Fold** | Wajib untuk klasifikasi; menjaga proporsi kelas di setiap fold |
| **cross_validate** | Evaluasi multiple metrics sekaligus; juga bisa return train scores |
| **Learning Curve** | Diagnosis underfitting vs overfitting; panduan apakah perlu lebih banyak data |
| **Validation Curve** | Efek satu hyperparameter; menemukan sweet spot regularisasi |
| **GridSearchCV** | Exhaustive; gunakan untuk ruang parameter kecil (<1000 kombinasi) |
| **RandomizedSearchCV** | Lebih efisien; gunakan untuk ruang parameter besar; sering sama baiknya |

### Workflow Evaluasi yang Benar
```
Data → [Train+Val | Test]
          ↓
    K-Fold CV pada Train+Val
          ↓
    Hyperparameter tuning
          ↓
    Final model dilatih pada seluruh Train+Val
          ↓
    Evaluasi SEKALI pada Test set
```
